In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd



In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
inputs_dir = base_path / "Inputs"
# Where to save
output_dir = Path(base_path) / "Connectivity"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
land_use_path = inputs_dir / "2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use_path).copy() # Optional: if you want to preserve the original
terrestrial_landcover.crs

In [ ]:
terrestrial_landcover["Classify"].unique()

In [ ]:
# mapping from land-cover class to factor
class_condition_factor_map = {
    "Primary vegetation, forest and non-forest": 1.0,
    "Secondary vegetation: mature (50 years and more)": 0.99,
    "Secondary vegetation: intermediate (30-50 years)": 0.855,
    "Secondary vegetation (young) <30 years)": 0.757,
    "Perennial croplands (C3 and C4)": 0.635,
    "Nitrogen fixing croplands": 0.585,
    "Annual croplands (C3 and C4)": 0.46,
    "Rangelands": 0.55,
    "Managed pasture": 0.533,
    "Urban": 0.645,
    "Zero_value": 0
}


In [ ]:
baseline_condition_types = {
    "Disturbed broadleaved forest (Secondary Forest)": "Secondary vegetation (young) <30 years)",  
    "Fields: Herbaceous crops, fallow, cultivated vegetables": "Annual croplands (C3 and C4)",
    "Secondary Forest": "Secondary vegetation: intermediate (30-50 years)",
    "Fields and Secondary Forest": {
        "Secondary vegetation (young) <30 years)": 0.5,
        "Annual croplands (C3 and C4)": 0.5,
    },
    "Buildings and other infrastructures": "Urban",
    "Closed broadleaved forest (Primary Forest)": "Primary vegetation, forest and non-forest",
    "Plantation: Tree crops, shrub crops, sugar cane, banana": "Perennial croplands (C3 and C4)",
    "Open dry forest - Tall (Woodland/Savanna)": "Primary vegetation, forest and non-forest",
    "Fields  and Bamboo": {   # (double space variant)
        "Annual croplands (C3 and C4)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Bamboo and Secondary Forest": {
        "Perennial croplands (C3 and C4)": 0.5,
        "Secondary vegetation: intermediate (30-50 years)": 0.5,
    },
    "Bamboo and Fields": {
        "Annual croplands (C3 and C4)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Herbaceous Wetland": "Primary vegetation, forest and non-forest",
    "Mangrove Forest": "Primary vegetation, forest and non-forest",
    "Fields or Secondary Forest/Pine Plantation": {
        "Annual croplands (C3 and C4)": 0.5, # I think this is where it had gone wrong - but I have corrected it
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Fields: Bare Land": "Managed pasture",
    "Fields: Pasture,Human disturbed, grassland": "Managed pasture",
    "Water Body": "Zero_value",
    "Bamboo": "Perennial croplands (C3 and C4)",
    "Bauxite Extraction": "Zero_value",
    "Open dry forest - Short": "Primary vegetation, forest and non-forest",
    "Bare Rock": "Zero_value",
    "Quarry": "Zero_value",
    "Hardwood Plantation: Mahogany": "Perennial croplands (C3 and C4)",
    "Swamp Forest": "Primary vegetation, forest and non-forest",
    "Hardwood Plantation: Euculytus": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mixed": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mahoe": "Perennial croplands (C3 and C4)",
}

In [ ]:
partial_reforest_no_plantations_bamboo = {
    "Disturbed broadleaved forest (Secondary Forest)": "Secondary vegetation: intermediate (30-50 years)",  
    "Fields: Herbaceous crops, fallow, cultivated vegetables": "Secondary vegetation (young) <30 years)",
    "Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Fields and Secondary Forest": {
        "Secondary vegetation (young) <30 years)": 0.5,
        "Secondary vegetation: intermediate (30-50 years)": 0.5,
    },
    "Buildings and other infrastructures": "Urban",
    "Closed broadleaved forest (Primary Forest)": "Primary vegetation, forest and non-forest",
    "Plantation: Tree crops, shrub crops, sugar cane, banana": "Perennial croplands (C3 and C4)",
    "Open dry forest - Tall (Woodland/Savanna)": "Primary vegetation, forest and non-forest",
    "Fields  and Bamboo": {   # (double space variant)
        "Secondary vegetation (young) <30 years)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Bamboo and Secondary Forest": {
        "Perennial croplands (C3 and C4)": 0.5,
        "Secondary vegetation: mature (50 years and more)": 0.5,
    },
    "Bamboo and Fields": {
        "Secondary vegetation (young) <30 years)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Herbaceous Wetland": "Primary vegetation, forest and non-forest",
    "Mangrove Forest": "Primary vegetation, forest and non-forest",
    "Fields or Secondary Forest/Pine Plantation": {
        "Secondary vegetation (young) <30 years)": 0.5, # I think this is where Its gone wrong - the other one had both perennial.
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Fields: Bare Land": "Secondary vegetation (young) <30 years)",
    "Fields: Pasture,Human disturbed, grassland": "Secondary vegetation (young) <30 years)",
    "Water Body": "Zero_value",
    "Bamboo": "Perennial croplands (C3 and C4)",
    "Bauxite Extraction": "Secondary vegetation (young) <30 years)",
    "Open dry forest - Short": "Primary vegetation, forest and non-forest",
    "Bare Rock": "Zero_value",
    "Quarry": "Secondary vegetation (young) <30 years)",
    "Hardwood Plantation: Mahogany": "Perennial croplands (C3 and C4)",
    "Swamp Forest": "Primary vegetation, forest and non-forest",
    "Hardwood Plantation: Euculytus": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mixed": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mahoe": "Perennial croplands (C3 and C4)",
}

In [ ]:
reforest_all_condition_types = {
    "Disturbed broadleaved forest (Secondary Forest)": "Secondary vegetation: intermediate (30-50 years)",  
    "Fields: Herbaceous crops, fallow, cultivated vegetables": "Secondary vegetation (young) <30 years)",
    "Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Fields and Secondary Forest": {
        "Secondary vegetation (young) <30 years)": 0.5,
        "Secondary vegetation: intermediate (30-50 years)": 0.5,
    },
    "Buildings and other infrastructures": "Urban",
    "Closed broadleaved forest (Primary Forest)": "Primary vegetation, forest and non-forest",
    "Plantation: Tree crops, shrub crops, sugar cane, banana": "Secondary vegetation (young) <30 years)",
    "Open dry forest - Tall (Woodland/Savanna)": "Primary vegetation, forest and non-forest",
    "Fields  and Bamboo": "Secondary vegetation (young) <30 years)",
    "Bamboo and Secondary Forest": {
        "Secondary vegetation (young) <30 years)": 0.5,
        "Secondary vegetation: mature (50 years and more)": 0.5,
    },
    "Bamboo and Fields": "Secondary vegetation (young) <30 years)",
    "Herbaceous Wetland": "Primary vegetation, forest and non-forest",
    "Mangrove Forest": "Primary vegetation, forest and non-forest",
    "Fields or Secondary Forest/Pine Plantation": "Secondary vegetation (young) <30 years)",
    "Fields: Bare Land": "Secondary vegetation (young) <30 years)",
    "Fields: Pasture,Human disturbed, grassland": "Secondary vegetation (young) <30 years)",
    "Water Body": "Zero_value",
    "Bamboo": "Secondary vegetation (young) <30 years)",
    "Bauxite Extraction": "Secondary vegetation (young) <30 years)",
    "Open dry forest - Short": "Primary vegetation, forest and non-forest",
    "Bare Rock": "Zero_value",
    "Quarry": "Secondary vegetation (young) <30 years)",
    "Hardwood Plantation: Mahogany": "Secondary vegetation (young) <30 years)",
    "Swamp Forest": "Primary vegetation, forest and non-forest",
    "Hardwood Plantation: Euculytus": "Secondary vegetation (young) <30 years)",
    "Hardwood Plantation: Mixed": "Secondary vegetation (young) <30 years)",
    "Hardwood Plantation: Mahoe": "Secondary vegetation (young) <30 years)",
}

In [ ]:
long_timeframe_condition_types_keep_plantations = {
    "Disturbed broadleaved forest (Secondary Forest)": "Secondary vegetation: mature (50 years and more)",  
    "Fields: Herbaceous crops, fallow, cultivated vegetables": "Secondary vegetation: mature (50 years and more)",
    "Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Fields and Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Buildings and other infrastructures": "Urban",
    "Closed broadleaved forest (Primary Forest)": "Primary vegetation, forest and non-forest",
    "Plantation: Tree crops, shrub crops, sugar cane, banana": "Perennial croplands (C3 and C4)",
    "Open dry forest - Tall (Woodland/Savanna)": "Primary vegetation, forest and non-forest",
    "Fields  and Bamboo": {   # (double space variant)
        "Secondary vegetation: mature (50 years and more)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Bamboo and Secondary Forest": {
        "Perennial croplands (C3 and C4)": 0.5,
        "Secondary vegetation: mature (50 years and more)": 0.5,
    },
    "Bamboo and Fields": {
        "Secondary vegetation: mature (50 years and more)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Herbaceous Wetland": "Primary vegetation, forest and non-forest",
    "Mangrove Forest": "Primary vegetation, forest and non-forest",
    "Fields or Secondary Forest/Pine Plantation": {
        "Secondary vegetation: mature (50 years and more)": 0.5,
        "Perennial croplands (C3 and C4)": 0.5,
    },
    "Fields: Bare Land": "Secondary vegetation: mature (50 years and more)",
    "Fields: Pasture,Human disturbed, grassland": "Secondary vegetation: mature (50 years and more)",
    "Water Body": "Zero_value",
    "Bamboo": "Perennial croplands (C3 and C4)",
    "Bauxite Extraction": "Secondary vegetation: mature (50 years and more)",
    "Open dry forest - Short": "Primary vegetation, forest and non-forest",
    "Bare Rock": "Zero_value",
    "Quarry": "Secondary vegetation: mature (50 years and more)",
    "Hardwood Plantation: Mahogany": "Perennial croplands (C3 and C4)",
    "Swamp Forest": "Primary vegetation, forest and non-forest",
    "Hardwood Plantation: Euculytus": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mixed": "Perennial croplands (C3 and C4)",
    "Hardwood Plantation: Mahoe": "Perennial croplands (C3 and C4)",
}

In [ ]:
long_timeframe_condition_types = {
    "Disturbed broadleaved forest (Secondary Forest)": "Secondary vegetation: mature (50 years and more)",  
    "Fields: Herbaceous crops, fallow, cultivated vegetables": "Secondary vegetation: mature (50 years and more)",
    "Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Fields and Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Buildings and other infrastructures": "Urban",
    "Closed broadleaved forest (Primary Forest)": "Primary vegetation, forest and non-forest",
    "Plantation: Tree crops, shrub crops, sugar cane, banana": "Secondary vegetation: mature (50 years and more)",
    "Open dry forest - Tall (Woodland/Savanna)": "Primary vegetation, forest and non-forest",
    "Fields  and Bamboo": "Secondary vegetation: mature (50 years and more)",
    "Bamboo and Secondary Forest": "Secondary vegetation: mature (50 years and more)",
    "Bamboo and Fields": "Secondary vegetation: mature (50 years and more)",
    "Herbaceous Wetland": "Primary vegetation, forest and non-forest",
    "Mangrove Forest": "Primary vegetation, forest and non-forest",
    "Fields or Secondary Forest/Pine Plantation": "Secondary vegetation: mature (50 years and more)",
    "Fields: Bare Land": "Secondary vegetation: mature (50 years and more)",
    "Fields: Pasture,Human disturbed, grassland": "Secondary vegetation: mature (50 years and more)",
    "Water Body": "Zero_value",
    "Bamboo": "Secondary vegetation: mature (50 years and more)",
    "Bauxite Extraction": "Secondary vegetation: mature (50 years and more)",
    "Open dry forest - Short": "Primary vegetation, forest and non-forest",
    "Bare Rock": "Zero_value",
    "Quarry": "Secondary vegetation: mature (50 years and more)",
    "Hardwood Plantation: Mahogany": "Secondary vegetation: mature (50 years and more)",
    "Swamp Forest": "Primary vegetation, forest and non-forest",
    "Hardwood Plantation: Euculytus": "Secondary vegetation: mature (50 years and more)",
    "Hardwood Plantation: Mixed": "Secondary vegetation: mature (50 years and more)",
    "Hardwood Plantation: Mahoe":"Secondary vegetation: mature (50 years and more)",
}

In [ ]:
def to_factor(x):
    if isinstance(x, dict):
        return sum(w * class_condition_factor_map[k] for k, w in x.items())
    return class_condition_factor_map.get(x, 0.0)

violations = []
for lu in sorted(set(baseline_condition_types) | set(reforest_all_condition_types) | set(partial_reforest_no_plantations_bamboo)):
    b = to_factor(baseline_condition_types.get(lu))
    p = to_factor(partial_reforest_no_plantations_bamboo.get(lu))
    r = to_factor(reforest_all_condition_types.get(lu))
    if p < b - 1e-12 or p > r + 1e-12:
        violations.append((lu, b, p, r))

if violations:
    for lu, b, p, r in violations:
        print(f"VIOLATION: {lu}  baseline={b:.3f}  partial={p:.3f}  reforest={r:.3f}")
else:
    print("All good: partial is between baseline and reforest for all classes.")

In [ ]:
# Unique land-use classes
classes = sorted(terrestrial_landcover["Classify"].dropna().unique())

# List your scenarios: (filename_suffix, mapping_dict)
scenarios = [
    ("baseline", baseline_condition_types),
    ("partial_no_plantations_bamboo", partial_reforest_no_plantations_bamboo),
    ("reforest_all", reforest_all_condition_types),
    ("long timeframe keep_plantations", long_timeframe_condition_types_keep_plantations), 
    ("long_timeframe", long_timeframe_condition_types),
]

# --- Build & save a CSV for each scenario, with Classify as the key column ---
for scenario_name, mapping in scenarios:
    rows = []
    for Classify in classes:  # value from terrestrial_landcover["Classify"]
        cond = mapping.get(Classify)  # string, dict, or None

        if isinstance(cond, dict):
            condition_name = "; ".join(f"{name} ({weight:g})" for name, weight in cond.items())
            factor = sum(weight * class_condition_factor_map.get(name, 0.0)
                         for name, weight in cond.items())
        else:
            condition_name = cond  # may be "Zero_value" or None
            factor = class_condition_factor_map.get(cond, 0.0)

        rows.append((Classify, condition_name, factor, float(factor) ** 4))

    condition_table = pd.DataFrame(
        rows,
        columns=["Classify", "condition_name", "condition_factor", "condition_factor_p4"]
    )

    out_csv = output_dir / f"landuse_condition_factors__{scenario_name}_p4.csv"
    condition_table.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

In [ ]:
# --- All ones ---
ones = 1.0
normalized_ones = pd.DataFrame({
    "Classify": classes,
    "condition_name": ["Normalized (ones)"] * len(classes),
    "condition_factor": [ones] * len(classes),
    "condition_factor_p4": [ones ** 4] * len(classes),   # = 1.0
})
ones_csv = output_dir / "landuse_condition_factors__normalized_ones_p4.csv"
normalized_ones.to_csv(ones_csv, index=False)
print("Saved:", ones_csv)


# Pick a tiny-but-safe epsilon so epsilon**4 is still > 0 in float64 and not rounded to 0 in Excel
epsilon = 1e-8            # -> p4 = 1e-32 (tiny, nonzero, safe)
epsilon_p4 = float(epsilon) ** 4

normalized_zeros = pd.DataFrame({
    "Classify": classes,
    "condition_name": [f"Normalized ({epsilon})"] * len(classes),
    "condition_factor": [epsilon] * len(classes),
    "condition_factor_p4": [epsilon_p4] * len(classes),
})

zeros_csv = output_dir / "landuse_condition_factors__normalized_zeros_p4.csv"
normalized_zeros.to_csv(zeros_csv, index=False, float_format="%.15e")
print("Saved:", zeros_csv)